# Analyse II: Alignierte Embeddings

Nächste Nachbarn der Zielwörter in den alignierten Zeitscheiben und ihre Veränderung über die Zeit.

In [1]:
import polars as pl

from zeitenwende_sgns.config import ALIGNED_DIR
from zeitenwende_sgns.neighbors import load_slices, neighbors_over_time
from zeitenwende_sgns.targets import load_targets

pl.Config.set_tbl_rows(30)

SOURCE = "cpp_bt"
PERIOD = "year"

slices = load_slices(ALIGNED_DIR / SOURCE / PERIOD)
targets = load_targets()
list(slices)

['2014',
 '2015',
 '2016',
 '2017',
 '2018',
 '2019',
 '2020',
 '2021',
 '2022',
 '2023',
 '2024',
 '2025']

## Abdeckung der Zielwörter

In wie vielen Zeitscheiben jedes Zielwort im Vokabular ist (Mindestfrequenz beim Training).

In [2]:
coverage = pl.DataFrame(
    [
        {
            "target": target,
            **{label: target in wv.key_to_index for label, wv in slices.items()},
        }
        for target in targets
    ]
).with_columns(slices=pl.sum_horizontal(pl.exclude("target")))

coverage["slices"].value_counts().sort("slices", descending=True)

slices,count
u32,u32
12,256
11,44
10,32
9,25
8,26
7,30
6,27
5,27
4,25


In [3]:
coverage.filter(pl.col("slices") < len(slices)).sort("slices")

target,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,slices
str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,u32
"""Spezialoperation""",false,false,false,false,false,false,false,false,false,false,false,false,0
"""Russland-Ukraine-Konflikt""",false,false,false,false,false,false,false,false,false,false,false,false,0
"""Russland-Ukraine-Krieg""",false,false,false,false,false,false,false,false,false,false,false,false,0
"""Cyberkrieg""",false,false,false,false,false,false,false,false,false,false,false,false,0
"""Heroisierung""",false,false,false,false,false,false,false,false,false,false,false,false,0
"""Kanonenfutter""",false,false,false,false,false,false,false,false,false,false,false,false,0
"""Beistandspflicht""",false,false,false,false,false,false,false,false,false,false,false,false,0
"""Kriegspräsident""",false,false,false,false,false,false,false,false,false,false,false,false,0
"""pazifistisch""",false,false,false,false,false,false,false,false,false,false,false,false,0


## Nachbarn je Zeitscheibe

In [11]:
WORD = "Angriffskrieg"

neighbors_over_time(slices, WORD, n=10).pivot("label", index="rank", values="neighbor")

rank,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
i64,str,str,str,str,str,str,str,str,str,str,str,str
1,"""Unregelmäßigkeit""","""Mordserie""","""Aufstacheln""","""steuerfinanzierten""","""Strafverfahren""","""völkerrechtswidrige""","""Fahndung""","""Prozedur""","""Ukraine""","""Russland""","""russisch""","""brutal"""
2,"""Elfenbeinhandel""","""krönen""","""8d""","""Knopf""","""Klimakonferenz""","""Erdogans""","""annektiert""","""Investmentfonds""","""Russland""","""Ukraine""","""Krieg""","""völkerrechtswidrigen"""
3,"""Führungsstab""","""Bevölkerungsteil""","""80a""","""meldepflichtig""","""beschuldigen""","""bombardiert""","""Verfahrensregister""","""Abriss""","""völkerrechtswidrigen""","""russisch""","""Iran""","""Russland"""
4,"""Jahresgutachten""","""Vorbereitungshandlung""","""8c""","""sowjetisch""","""Unvereinbarkeit""","""mörderisch""","""Assad-Regime""","""unverhohlen""","""Krieg""","""Krieg""","""Angriff""","""russisch"""
5,"""vorzeichnen""","""unweigerlich""","""233""","""unmenschlich""","""556""","""Kriegsverbrecher""","""Demos""","""Wehr""","""brutal""","""Putin""","""brutal""","""Krieg"""
6,"""Volksinitiative""","""rechtsextremistischen""","""Ff""","""Biodiesel""","""erster""","""verbrecherisch""","""Klassiker""","""kriminalisieren""","""russisch""","""brutal""","""Überfall""","""Terror"""
7,"""Ehrenmal""","""Fanatismus""","""179""","""Bevorrechtigung""","""Politische""","""YPG""","""Jinping""","""Strafnorm""","""Putin""","""völkerrechtswidrigen""","""Russland""","""Angriff"""
8,"""diktiert""","""wirkend""","""Körperschaftsteuergesetz""","""Verfall""","""Ausarbeitung""","""Einmarsch""","""linksextremistische""","""absolvieren""","""Angriff""","""Angriff""","""Hisbollah""","""Gaza"""
9,"""Vorbedingung""","""Aktivist""","""Tatbestand""","""Stalin""","""Infragestellen""","""Nordostsyrien""","""Sozialgericht""","""Studies""","""Wladimir""","""Hamas""","""Terror""","""Kriegsverbrechen"""
